<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Training versus inference

**[Training versus inference](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/)** · Deep Learning and Transformers Explained · Module 7, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** load a small pretrained language model, read its architecture and shapes against the transformer block, and run the generation loop one token at a time, by hand and with `generate()`.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 270 MB for the model, 2 MB for the tickets). No account, no graphics card. About 600 MB of free memory. |
| **You should know** | The transformer block of [Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/), the causal mask of [Position and masking](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/) and next-token prediction from [Training objectives](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled; Hugging Face transformers 5.19.0 |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M07-L01-training-versus-inference/training-versus-inference-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. It also installs Hugging Face `transformers` for the small pretrained model SmolLM2-135M (about 270 MB to download later).

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
%pip install -q --disable-pip-version-check "transformers>=5,<6" safetensors
import transformers; print("transformers", transformers.__version__)
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: load SmolLM2-135M

Run the next cell. It downloads the small pretrained model **SmolLM2-135M** from Hugging Face and loads it. The model is free and open (licence Apache-2.0), and it needs no account.

- The first run downloads about **270 MB**. The makers store the weights in the 16-bit format bfloat16, 2 bytes per weight. The download can take a minute; later runs use the saved copy.
- The cell loads the weights as **float32**, 4 bytes per weight, so the model takes about **540 MB** of memory. The lessons' numbers were measured in float32, and every CPU computes float32 well. [Inference costs](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/) explains the difference.
- `revision=` asks for one exact version of the model files, so your model is the course's model, even if the makers update theirs.

You should see a progress bar, then `parameters: 134515008`.

If the cell shows an error:

- `OSError` or `Connection`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `No module named 'transformers'`: run the first setup cell again.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-135M"
REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"  # the exact version of the files that the course used
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.float32)
model.eval()                                   # inference: switch off training-only behaviour
PROMPT = ("Larkfield is an online shop for home and garden products. "
          "A customer wrote: my parcel is late. The support team")
print("parameters:", sum(p.numel() for p in model.parameters()))

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Read the architecture

`print(model)` lists every part of the network, in order. Compare it with the transformer block that you know: attention, add and normalize, feed-forward, add and normalize.

> **Predict.** The model has 30 blocks. Will `print(model)` show 30 separate blocks, or one block with "30 ×"? Then run the cell.

In [ ]:
print(model)

> **Check.** You should see `LlamaForCausalLM`, then `embed_tokens: Embedding(49152, 576)`, then `(0-29): 30 x LlamaDecoderLayer`, and at the end `lm_head: Linear(in_features=576, out_features=49152)`.

Read one block from the top: `self_attn` (attention), `mlp` (the feed-forward layer), and two `LlamaRMSNorm` layers (a simpler layer normalization). The additions of the residual connections are in the code of the block, so `print` does not show them.

Look at `k_proj` and `v_proj`: they give 192 numbers per token, not 576. The model has 9 query heads of 64 numbers (9 × 64 = 576) but only 3 key-value heads (3 × 64 = 192). Each key-value head serves a group of 3 query heads: **grouped-query attention**.

The next cell counts the parameters of each part, and checks that the output layer `lm_head` uses the same matrix as the embedding (**tied embeddings**).

In [ ]:
block = model.model.layers[0]
count = lambda module: sum(p.numel() for p in module.parameters())
print("embedding  ", count(model.model.embed_tokens))
print("one block  ", count(block))
print("  attention", count(block.self_attn))
print("  mlp      ", count(block.mlp))
print("30 blocks  ", sum(count(b) for b in model.model.layers))
print("lm_head is the embedding:", model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr())

> **Check.** You should see `embedding 28311552`, `one block 3540096`, `attention 884736`, `mlp 2654208`, `30 blocks 106202880` and `lm_head is the embedding: True`.

> **Your turn.** Compute the attention parameters of one block by hand from the four shapes: `q_proj` 576 → 576, `k_proj` 576 → 192, `v_proj` 576 → 192, `o_proj` 576 → 576 (no biases). Put the number in `attention_by_hand`, then run the check cell.

In [ ]:
attention_by_hand = ...  # four matrices: inputs x outputs each

In [ ]:
expect('attention_by_hand', attention_by_hand, 884736)

## 2. Tokens and shapes

The tokenizer cuts the text into pieces from the model's vocabulary of 49,152 tokens. `Ġ` marks a space before a piece.

> **Predict.** How many tokens does the prompt have, and what is the shape of the logits (the scores) that the model returns? Then run the cell.

In [ ]:
inputs = tokenizer(PROMPT, return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
show_shape("input_ids", inputs["input_ids"])
with torch.no_grad():                          # inference: no record for gradients
    logits = model(**inputs).logits
show_shape("logits", logits)

> **Check.** You should see 25 pieces, from `'L', 'ark', 'field', 'Ġis'` to `'Ġsupport', 'Ġteam'`, then `input_ids shape (1, 25)` and `logits shape (1, 25, 49152)`.

The logits have one row of 49,152 scores **for every position**: at each position, the model predicts the token after it. Only the last row predicts the next token after the whole prompt. Softmax turns it into probabilities.

In [ ]:
probs = torch.softmax(logits[0, -1], dim=-1)  # the last row -> probabilities
top = torch.topk(probs, 8)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode([int(i)])!r:12} {p.item():.4f}")

> **Check.** You should see `' is'` with about `0.1069`, then `' has'` with `0.0891` and `' will'` with `0.0471`. No token is very likely: many continuations are good English.

## 3. The generation loop, by hand

Generation repeats one step: run the model, take the most likely next token (greedy choice), add it to the text, and run again. The loop below does this 10 times. It runs the model on the **whole** text at every step.

> **Predict.** Will the 10 tokens of this loop be the same as the 10 tokens of `model.generate()` with greedy choice? Then run the two cells.

In [ ]:
ids = inputs["input_ids"]
chosen = []
with torch.no_grad():
    for step in range(10):
        logits = model(input_ids=ids).logits[0, -1]       # scores for the next token
        next_id = int(logits.argmax())                     # greedy: the most likely token
        chosen.append(next_id)
        ids = torch.cat([ids, torch.tensor([[next_id]])], dim=1)
        print(f"step {step + 1:2}   input {ids.shape[1] - 1} tokens   chosen {tokenizer.decode([next_id])!r}")

In [ ]:
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
new_ids = out[0, inputs["input_ids"].shape[1]:].tolist()
print("generate():", repr(tokenizer.decode(new_ids)))
print("same tokens as the loop:", new_ids == chosen)

> **Check.** The loop chooses `' is'`, `' very'`, `' helpful'`, `'.'`, `' I'`, `' am'`, `' looking'`, `' for'`, `' a'`, `' new'`. Then you should see `same tokens as the loop: True`.

`generate()` runs the same loop. It is faster, because it keeps a **KV cache**: it does not compute the earlier tokens again at every step. The next lesson measures the difference.

## 4. Greedy choice repeats itself

> **Predict.** What will 40 greedy tokens look like? Then run the cell.

In [ ]:
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
greedy_text = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:])
print(repr(greedy_text))

> **Check.** You should see `' is very helpful. I am looking for a new parcel. I am looking for a new parcel. …'`: the same sentence again and again.

Once a sentence repeats, the repeated text makes the same sentence even more likely, and greedy choice always takes it. SmolLM2-135M is a **base model**: it was only pretrained to continue text, so it continues the customer's message instead of answering it.

## 5. Sampling with a temperature

Sampling draws the next token at random, with the model's probabilities as the chances. The **temperature** re-weights the probabilities first: below 1 the likely tokens get more, above 1 the shares become more even. A **seed** fixes the random draws, so that you can repeat them. `top_p=0.95` keeps only the most likely tokens that together have 95% of the probability.

In [ ]:
for seed in (1, 2, 3):
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=25, do_sample=True, temperature=0.8, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    print(seed, repr(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:])))

> **Check.** You should see three different continuations. On the course's computer, seed 1 starts with `' is currently at my front door'`. On another kind of processor, the sampled text can differ even with the same seed, because tiny rounding differences change a draw.

## 6. Can a base model route a ticket?

Prompting is one way to adapt a pretrained model: you write instructions and examples into the text, and the weights stay the same. The cell below gives SmolLM2 the five teams, then one ticket, then `Team:`. It asks two questions: what does the model write (greedy), and which of the five team names does it rate most likely?

In [ ]:
TEAM_IDS = [tokenizer(" " + team)["input_ids"][0] for team in TEAMS]   # each team name is one token
INSTRUCTION = "Each support ticket goes to one team: delivery, returns, payment, warranty or account.\n\n"

def route(prefix, text):
    """The team whose name the model rates most likely after 'Team:'."""
    ids = tokenizer(prefix + f"Ticket: {text}\nTeam:", return_tensors="pt")
    with torch.no_grad():
        logits = model(**ids).logits[0, -1]
    return TEAMS[int(logits[TEAM_IDS].argmax())]

ticket = valid_rows[0]
ids = tokenizer(INSTRUCTION + f"Ticket: {ticket['text']}\nTeam:", return_tensors="pt")
with torch.no_grad():
    out = model.generate(**ids, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print("real team:", ticket["team"])
print("the model writes:", repr(tokenizer.decode(out[0, ids["input_ids"].shape[1]:])))
print("most likely team name:", route(INSTRUCTION, ticket["text"]))

> **Check.** The real team is `payment`. The model writes `' I am writing regarding order LK-714258. The electric kettle was …'`: it continues the ticket instead of naming a team.

A base model does not follow instructions; it continues text. Examples in the prompt (**few-shot** prompting) show it the pattern. The next cell adds one example ticket for each team and scores the first 100 validation tickets both ways. It takes about 10 to 30 seconds.

> **Predict.** The keyword rule of Module 1 routes 57.3% of the validation tickets right, and TF-IDF with logistic regression 98.8%. Where will SmolLM2 land with five examples?

In [ ]:
shots = []
for team in TEAMS:  # one short, plain training ticket per team
    plain = [r for r in train_rows[:400] if r["team"] == team and r["style"] == "plain"]
    shots.append(min(plain, key=lambda r: len(r["text"])))
FEW_SHOT = INSTRUCTION + "".join(f"Ticket: {r['text']}\nTeam: {r['team']}\n\n" for r in shots)

first_100 = valid_rows[:100]
zero_right = sum(route(INSTRUCTION, r["text"]) == r["team"] for r in first_100)
few_right = sum(route(FEW_SHOT, r["text"]) == r["team"] for r in first_100)
print("zero-shot, right:", zero_right, "of 100")
print("few-shot,  right:", few_right, "of 100")

> **Check.** You should see about 44 of 100 right without examples and about 45 with five examples (one or two more or fewer on another processor).

Both are far below the 98.8% of TF-IDF with logistic regression. A 135-million-parameter base model is not a ticket router. To make a pretrained model good at one task, you **fine-tune** it on labelled examples: the case study [Fine-tune a small transformer](https://learning.nextia-ai.com/courses/deep-learning/m09/fine-tune-a-small-transformer/) measures what that buys.

> **Your turn.** Count how the few-shot prompt spreads its answers over the five teams. Put a dictionary `{team: number of tickets}` in `few_counts`, then run the check cell.

In [ ]:
few_counts = ...  # {team: how many of the 100 tickets the few-shot prompt sends there}

> **Check.** Run the next cell. It checks that the counts add up to 100 and names the team that gets the fewest tickets.

In [ ]:
if few_counts is Ellipsis or not isinstance(few_counts, dict):
    print("Not yet: few_counts has no value. Write your answer in the cell above, then run it again.")
elif sum(few_counts.values()) != 100:
    print(f"Not yet: the counts add up to {sum(few_counts.values())}, not 100.")
else:
    print("Check passed. The fewest tickets go to:", min(few_counts, key=few_counts.get))
    print("The first 100 validation tickets have 30 delivery, 19 returns, 20 payment, 15 warranty and 16 account.")

On the course's computer, the few-shot prompt sends only 2 tickets to delivery, the largest team, and 34 to payment. The examples changed which team the model favours, not how well it reads the tickets.

## Next

You can now load a pretrained model, read its parts and shapes, and run its generation loop by hand. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/) has the knowledge checks. The next lesson, [Inference costs](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/), measures what this loop costs in memory and time.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Inference costs](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.